# Chapter 19: Carrying Out an Empirical Project

Topics: organising Python scripts, logging output, Jupyter notebooks for reproducible research.

This notebook serves as a **template** for an end-to-end empirical project.

## 19.1 Project Template Structure

```
project/
├── data/          ← raw data files
├── notebooks/     ← Jupyter notebooks (this file)
├── output/        ← figures, tables, logs
└── README.md
```

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import wooldridge as woo
import matplotlib.pyplot as plt
import os, datetime

output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

# Log output with timestamp
log_file = os.path.join(output_dir, "project_log.txt")
def log(msg):
    ts = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{ts}] {msg}"
    print(line)
    with open(log_file, 'a') as f:
        f.write(line + "\n")

log("Project started")

## 19.2 Full Empirical Example: Returns to Education

In [ ]:
# --- Data ---
df = woo.dataWoo('wage1')
log(f"Dataset loaded: {df.shape[0]} observations, {df.shape[1]} variables")

# --- Descriptive statistics ---
desc = df[['wage','educ','exper','tenure','female']].describe()
print(desc)
desc.to_csv(os.path.join(output_dir, "descriptive_stats.csv"))
log("Descriptive stats saved")

In [ ]:
# --- Regression ---
model = smf.ols('np.log(wage) ~ educ + exper + I(exper**2) + tenure + female', data=df).fit()
print(model.summary())

# Save results table
results_df = pd.DataFrame({
    'coef': model.params,
    'std_err': model.bse,
    't': model.tvalues,
    'p': model.pvalues,
    'CI_low': model.conf_int()[0],
    'CI_high': model.conf_int()[1]
})
results_df.to_csv(os.path.join(output_dir, "regression_results.csv"))
log("Regression results saved")

In [ ]:
# --- Diagnostics ---
from statsmodels.stats.diagnostic import het_breuschpagan, linear_reset
from statsmodels.stats.stattools import durbin_watson

bp = het_breuschpagan(model.resid, model.model.exog)
dw = durbin_watson(model.resid)
reset = linear_reset(model, power=2, use_f=True)

diagnostics = {
    'Breusch-Pagan p-value': bp[1],
    'Durbin-Watson':         dw,
    'RESET p-value':         reset.pvalue
}
for k, v in diagnostics.items():
    log(f"{k}: {v:.4f}")

In [ ]:
# --- Final Figure ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(df['educ'], np.log(df['wage']), alpha=0.3, color='steelblue')
axes[0].plot(df['educ'], model.fittedvalues - (model.params['exper']*df['exper'].mean())
             - (model.params['I(exper ** 2)']*df['exper'].mean()**2)
             - (model.params['tenure']*df['tenure'].mean()), 'r-', lw=2)
axes[0].set_xlabel('Education'); axes[0].set_ylabel('Log(Wage)')
axes[0].set_title('Education and Log Wages')

axes[1].scatter(model.fittedvalues, model.resid, alpha=0.4, color='steelblue')
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_xlabel('Fitted Values'); axes[1].set_ylabel('Residuals')
axes[1].set_title('Residual Plot')

plt.tight_layout()
fig.savefig(os.path.join(output_dir, "final_figure.png"), dpi=150)
log("Final figure saved")
plt.show()

log("Project completed")
print(f"\nAll output saved to: {output_dir}")